In [1]:
%pip install numpy
%pip install imageio
%pip install pygraphviz
%pip install matplotlib
%pip install networkx
%pip install tqdm


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgr

In [2]:
from collections import deque
import pygraphviz
import imageio
import numpy as np 
import matplotlib.pyplot as plt
import networkx as nx
from tqdm import tqdm

In [3]:
class QueensBoardChess:

    def __init__(self, size_board: int):
        self.n = size_board
        self.board = np.zeros((self.n, self.n))
    
    def position_queen(self, row: int, column: int) -> bool:
        if 0 <= row < self.n and 0 <= column < self.n:
            self.board[row][column] = 1
            return True
        else:
            return False
    
    def remove_queen(self, row: int, column: int) -> bool:
        if 0 <= row < self.n and 0 <= column < self.n:
            self.board[row][column] = 0
            return True
        else:
            return False
    
    def valid_position(self, row: int, column: int) -> bool:
        for i in range(row):
            if self.board[i][column] == 1:
                return False

        i, j = row, column
        while i >= 0 and j >= 0:
            if self.board[i][j] == 1:
                return False
            i -= 1
            j -= 1

        k, y = row, column
        while k >= 0 and y < self.n:
            if self.board[k][y] == 1:
                return False
            k -= 1
            y += 1

        return True

In [4]:
class Problem:

    def __init__(self, initial_state, actions, transition_model, goal_test, step_cost):

        self.initial_state = initial_state
        self.actions = actions                    
        self.transition_model = transition_model  
        self.goal_test = goal_test                
        self.step_cost = step_cost  

class Node:

    def __init__(self, problem, parent = None, action = None):

        self.parent = parent
        self.action = action

        if parent is None:
            self.state = problem.initial_state
            self.path_cost = 0.0
        else:
            self.state = problem.transition_model(parent.state, action)
            self.path_cost = parent.path_cost + problem.step_cost(parent.state, action)


    def __eq__(self, other):

        return isinstance(other, Node) and self.state == other.state

    def __hash__(self):

        return hash(self.state)

    def solution(self):

        node = self
        path = []

        while node.parent is not None:
            path.append(node.action)
            node = node.parent
        path.reverse()
        
        return path

In [5]:
class QueensProblem(Problem):

    def __init__(self, start_node, size_board: int):

        self.board_chess = QueensBoardChess(size_board)

        super().__init__(initial_state = start_node,
                         actions = self.actions_fn,
                         transition_model = self.transition_fn,
                         goal_test = self.goal_test_fn,
                         step_cost = self.step_cost_fn)

    def actions_fn(self, state):

        actions = []
        current_row = len(state)

        if current_row >= self.board_chess.n:
            return actions

        self.board_chess.board.fill(0)
        for each_row, each_column in enumerate(state):
            self.board_chess.position_queen(each_row, each_column)

        for column in range(self.board_chess.n):
            if self.board_chess.valid_position(current_row, column):
                actions.append((current_row, column))  
        return actions

    def transition_fn(self, state, action):

        row, col = action
        return state + (col,)

    def goal_test_fn(self, state):
            
        return len(state) == self.board_chess.n

    def step_cost_fn(self, state, action, uniform_cost = True):
        return 1

In [6]:
def state_to_board_str(state, n):
    grid = [["." for _ in range(n)] for _ in range(n)]
    for r, c in enumerate(state):
        grid[r][c] = "Q"
    return "\n".join(" ".join(row) for row in grid)

In [36]:
class BreadthFirstSearch:

    def __init__(self, problem):

        self.problem = problem
        self.frontier = deque()  
        self.explored = set()    
        self.tree_graph = nx.DiGraph()
        self.history = []

    def search(self):

        root = Node(problem = self.problem)
        n = self.problem.board_chess.n

        board_label = state_to_board_str(root.state, n)
        self.tree_graph.add_node(id(root), state = str(root.state), label = board_label)

        self.frontier.append(root)

        with tqdm(desc = "Executando BFS", unit = " nó") as pbar:
            while self.frontier:

                node = self.frontier.popleft()
                self.explored.add(node)

                active_nodes = list(self.tree_graph.nodes())
                self.history.append((active_nodes, id(node)))

                if self.problem.goal_test(node.state):
                    return node

                for action in self.problem.actions(node.state):
                    child = Node(self.problem, node, action)

                    child_label = state_to_board_str(child.state, n)
                    self.tree_graph.add_node(id(child), state = str(child.state), label=child_label)
                    self.tree_graph.add_edge(id(node), id(child))

                    if (child not in self.explored) and (child not in self.frontier):
                        self.frontier.append(child)
                pbar.update(1)

        return None

In [35]:
class DepthFirstSearch:

    def __init__(self, problem):
        self.problem = problem
        self.frontier = []
        self.explored = set()
        self.tree_graph = nx.DiGraph()
        self.history = []

    def search(self):
        root = Node(problem = self.problem)
        n = self.problem.board_chess.n

        board_label = state_to_board_str(root.state, n)
        self.tree_graph.add_node(id(root), state = str(root.state), label = board_label)

        if self.problem.goal_test(root.state):
            return root

        self.frontier.append(root)

        with tqdm(desc = "Executando DFS", unit = " nó") as pbar:
            while self.frontier:
                node = self.frontier.pop()

                if node in self.explored:
                    continue

                self.explored.add(node)

                active_nodes = list(self.tree_graph.nodes())
                self.history.append((active_nodes, id(node)))

                if self.problem.goal_test(node.state):
                    return node

                for action in reversed(list(self.problem.actions(node.state))):
                    child = Node(self.problem, node, action)

                    child_label = state_to_board_str(child.state, n)
                    self.tree_graph.add_node(id(child), state = str(child.state), label = child_label)
                    self.tree_graph.add_edge(id(node), id(child))

                    if (child not in self.explored) and (child not in self.frontier):
                        self.frontier.append(child)
                pbar.update(1)

        return None

In [9]:
def generate_animation(graph, steps, filename = "n_queens_search.gif"):
    frames = []

    pos = nx.drawing.nx_agraph.graphviz_layout(graph, prog = "dot")
    fig, ax = plt.subplots(figsize = (22, 10))

    for step, (active_nodes, current_node) in tqdm(list(enumerate(steps))):
        ax.clear()
        subgraph = graph.subgraph(active_nodes)

        node_colors = [
            "lightgreen" if n == current_node else "white" 
            for n in subgraph.nodes()
        ]

        labels = {n: graph.nodes[n]["label"] for n in subgraph.nodes()}

        nx.draw(
            subgraph,
            pos,
            labels = labels,
            node_color = node_colors,
            node_shape = "s",        
            node_size = 3500,      
            font_size = 8,
            font_family = "monospace", 
            edgecolors = "black",   
            arrowsize = 12,
            ax = ax,
        )
        ax.set_title(f"Passo {step + 1}: Expansão da Árvore de Busca", fontsize = 12)

        fig.canvas.draw()
        image = imageio.core.util.Array(
            np.array(fig.canvas.buffer_rgba())
        )

        frames.append(image)

    plt.close()
    
    imageio.mimsave(filename, frames, fps = 1) 



In [99]:
problem_instance = QueensProblem(start_node = (), size_board = 4)

print("Breadth-First Search (BFS):")

bfs = BreadthFirstSearch(problem_instance)
solution_node = bfs.search()

if solution_node:
    print("Solução encontrada (colunas por linha):", solution_node.state)
print("Nós explorados (BFS):", len(bfs.explored))
generate_animation(bfs.tree_graph, bfs.history, filename="n_queens_bfs.gif")

print("\nDepth-First Search (DFS):")

dfs = DepthFirstSearch(problem_instance)
solution_node_dfs = dfs.search()

if solution_node_dfs:
    print("Solução DFS (colunas por linha):", solution_node_dfs.state)
print("Nós explorados (DFS):", len(dfs.explored))
generate_animation(dfs.tree_graph, dfs.history, filename="n_queens_dfs.gif")

Breadth-First Search (BFS):


Executando BFS: 15 nó [00:00, 78154.73 nó/s]


Solução encontrada (colunas por linha): (1, 3, 0, 2)
Nós explorados (BFS): 16


100%|██████████| 16/16 [00:00<00:00, 17.03it/s]



Depth-First Search (DFS):


Executando DFS: 8 nó [00:00, 53515.84 nó/s]


Solução DFS (colunas por linha): (1, 3, 0, 2)
Nós explorados (DFS): 9


100%|██████████| 9/9 [00:00<00:00, 29.39it/s]
